# GLM: Spatial Position vs Reward-Proximity Coding in V1

**Goal**: Dissociate spatial position coding from reward-proximity coding in
layer-specific calcium imaging data from mouse V1 during virtual reality navigation.

**Pipeline overview**
1. Data loading and alignment
2. Regressor construction (design matrix X)
3. Ridge regression with cross-validation (4 model variants per cell)
4. Partial R² computation and cell classification
5. Layer-specific summary statistics and figures

**Assumptions to verify before running**
- `speed_tmlog_cm_s` must be added to the HDF5 first by running `Calculate_Speed_From_Tmlog.py`
- Reward is delivered 1.5 s after each teleportation ('t' event / lap end)
- Landmarks are at 25, 55, 85, 115 cm along the 130 cm corridor
- Layer anatomy: L4 = peak cell density ±70 µm; L5 = L4 lower + 150 µm


---
## CHUNK 1 — Data loading and alignment

Load neural activity, position, speed (both VRlog and TMlog versions),
reward delivery times, and trial boundaries from the preprocessed HDF5.
All signals end up on a common timebase: the **concatenated lap frame array**
stored in the HDF5 (gaps between laps have been removed by preprocessing).

**Key variables produced**
| Variable | Shape | Description |
|---|---|---|
| `spikes` | (n_cells, n_frames) | Deconvolved spike estimates (smoothed) |
| `location_cm` | (n_frames,) | Position in cm |
| `speed_vrlog` | (n_frames,) | Speed from VRlog (diff of VR position) |
| `speed_tmlog` | (n_frames,) | Speed from treadmill encoder |
| `reward_frames` | (n_laps,) | Frame index of reward delivery per lap |
| `lap_starts` / `lap_ends` | (n_laps,) | Frame boundaries |
| `med_coords` | (n_cells, 2) | Cell [y, x] pixel coordinates |


In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
from helper import read_h5

# ─── USER SETTINGS ──────────────────────────────────────────────────────────
# Path to a single *_preproc.h5 file.
# For developmental trajectory (Chunk 5), set PREPROC_H5_PATHS to all 7 days.
PREPROC_H5_PATH = (
    r"D:\V1_SpatialModulation\2p\V1_prism\JSY052_ChronicImaging"
    r"\251009_JSY_JSY052_SpatialModulation_Day1"
    r"\TSeries-10092025-1542-002\251009_JSY052_preproc.h5"
)

# Which speed signal to use as the GLM nuisance regressor.
# 'tmlog'  → speed_tmlog_cm_s  (requires Calculate_Speed_From_Tmlog.py to have run)
# 'vrlog'  → speed_cm_s  (always available)
SPEED_SOURCE = 'tmlog'

# Corridor / reward geometry
TRACK_LENGTH_CM     = 130.0       # total corridor length
REWARD_DELAY_S      = 1.5         # reward arrives this many seconds after teleportation
LANDMARK_POSITIONS  = [25, 55, 85, 115]  # cm, all 4 landmarks
REWARD_ZONE_CM      = 115.0       # reward landmark position (≈ end of track)
# ─────────────────────────────────────────────────────────────────────────────

# ── Load HDF5 ────────────────────────────────────────────────────────────────
print(f"Loading: {os.path.basename(PREPROC_H5_PATH)}")
data = read_h5(PREPROC_H5_PATH)

spikes      = data['smoothed_spks_temporal'].astype(np.float64)  # (n_cells, n_frames)
location_cm = data['location_cm'].astype(np.float64)             # (n_frames,)
speed_vrlog = data['speed_cm_s'].astype(np.float64)              # (n_frames,)
lap_starts  = data['lap_starts'].astype(int)                     # (n_laps,)
lap_ends    = data['lap_ends'].astype(int)                        # (n_laps,)
med_coords  = data['med_coords'].astype(np.float64)              # (n_cells, 2)
bin_centers = data['bin_centers'].astype(np.float64)             # spatial bin centres
framerate   = float(data['processing_params']['framerate'])
n_cells, n_frames = spikes.shape
n_laps            = len(lap_starts)

# ── Load TMlog speed (if available) ─────────────────────────────────────────
with h5py.File(PREPROC_H5_PATH, 'r') as f:
    has_tmlog = 'speed_tmlog_cm_s' in f

if SPEED_SOURCE == 'tmlog':
    if has_tmlog:
        speed_tmlog = data['speed_tmlog_cm_s'].astype(np.float64)
        print("  Using TMlog speed (treadmill encoder).")
    else:
        print("  WARNING: TMlog speed not found in HDF5.")
        print("  Run Calculate_Speed_From_Tmlog.py first, or set SPEED_SOURCE='vrlog'.")
        print("  Falling back to VRlog speed.")
        speed_tmlog = speed_vrlog.copy()
else:
    speed_tmlog = speed_vrlog.copy()
    print("  Using VRlog-derived speed.")

# Speed used in GLM
speed = speed_tmlog

# ── Compute reward frame indices ─────────────────────────────────────────────
# Reward arrives REWARD_DELAY_S seconds after each lap end (teleportation).
# lap_ends[k] is the last frame of lap k in the concatenated array.
reward_delay_frames = int(np.round(REWARD_DELAY_S * framerate))
reward_frames = np.array([
    min(le + reward_delay_frames, n_frames - 1)
    for le in lap_ends
], dtype=int)

# ── Alignment verification ───────────────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"n_cells       : {n_cells}")
print(f"n_frames      : {n_frames}")
print(f"n_laps        : {n_laps}")
print(f"framerate     : {framerate:.2f} Hz")
print(f"recording dur : {n_frames / framerate:.1f} s")
print(f"position range: {location_cm.min():.1f} – {location_cm.max():.1f} cm")
print(f"speed range   : {speed.min():.1f} – {speed.max():.1f} cm/s")
print(f"reward frames : {len(reward_frames)} events")

# Check all arrays share the same time axis
assert spikes.shape[1] == n_frames, "spike/frame mismatch"
assert len(location_cm) == n_frames, "location/frame mismatch"
assert len(speed) == n_frames, "speed/frame mismatch"
print("\n✓ All dimensions verified.")

# ── Quick overview plot ───────────────────────────────────────────────────────
t = np.arange(n_frames) / framerate
fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
axes[0].plot(t, location_cm, lw=0.5, color='steelblue')
axes[0].set_ylabel('Position (cm)')
axes[0].set_title('Position and speed overview (first 300 s)')
for rw in reward_frames[reward_frames < int(300*framerate)]:
    axes[0].axvline(rw/framerate, color='red', alpha=0.3, lw=0.5)
axes[1].plot(t, speed, lw=0.5, color='darkorange', label=SPEED_SOURCE)
axes[1].plot(t, speed_vrlog, lw=0.5, color='steelblue', alpha=0.5, label='vrlog')
axes[1].set_ylabel('Speed (cm/s)')
axes[1].set_xlabel('Time (s)')
axes[1].legend(fontsize=8)
axes[0].set_xlim(0, min(300, t[-1]))
plt.tight_layout()
plt.show()


---
## CHUNK 2 — Regressor construction

Build a design matrix **X** (n_frames × n_regressors) containing:

| Group | Regressors | Description |
|---|---|---|
| Spatial | 10 raised-cosine basis fns | Tile 0–130 cm |
| Reward-proximity | 2 | Linear + quadratic distance to reward zone |
| Landmark | 4 | Per-landmark onset impulse × GCaMP kernel |
| Nuisance | 4 | z-speed, trial-onset impulse, trial-offset impulse, reward impulse |

### GCaMP kernel estimation
We cannot observe raw transients directly in deconvolved data, so we estimate
the effective kernel by computing the **reward-triggered average (RTA)** of the
raw neural activity.  Reward delivery is a hard, reliable event; the RTA
shape reflects the temporal blurring imposed by the GCaMP indicator and
the deconvolution step combined.

Steps:
1. For each reward frame, extract a ±K frame window of spike activity (averaged across all cells and all laps).
2. Take the causal half (t ≥ 0) and normalise to sum to 1 → this is the **kernel**.
3. Convolve each landmark-onset impulse train with this kernel.

The kernel encodes: *"if a neuron fires a spike, how does its Ca²⁺ signal
(and our deconvolved estimate of it) evolve over subsequent frames?"*


In [ ]:
from scipy.ndimage import gaussian_filter1d
from scipy.signal   import convolve

# ════════════════════════════════════════════════════════════════════════
# 2.1  GCaMP calcium kernel (reward-triggered average)
# ════════════════════════════════════════════════════════════════════════
KERNEL_WINDOW_S = 3.0   # look 3 s after reward delivery
KERNEL_PRE_S    = 0.5   # look 0.5 s before (to baseline-correct)

kernel_win_frames = int(np.round(KERNEL_WINDOW_S * framerate))
kernel_pre_frames = int(np.round(KERNEL_PRE_S    * framerate))
kernel_total      = kernel_pre_frames + kernel_win_frames

# Collect reward-triggered snippets (average across all cells)
rta_snippets = []
for rw in reward_frames:
    start = rw - kernel_pre_frames
    end   = rw + kernel_win_frames
    if start >= 0 and end <= n_frames:
        snippet = spikes[:, start:end].mean(axis=0)   # mean over cells
        rta_snippets.append(snippet)

rta = np.array(rta_snippets).mean(axis=0)  # (kernel_total,)

# Baseline-subtract using pre-reward window
baseline  = rta[:kernel_pre_frames].mean()
rta_clean = rta - baseline

# Take causal portion (t ≥ 0), clip negative values, normalise
kernel_raw      = rta_clean[kernel_pre_frames:]
kernel_raw      = np.maximum(kernel_raw, 0)
kernel_raw_sum  = kernel_raw.sum()
if kernel_raw_sum > 0:
    calcium_kernel = kernel_raw / kernel_raw_sum
else:
    # Fallback: synthetic double-exponential kernel (tau_rise=0.15 s, tau_decay=0.6 s)
    print("  WARNING: RTA is flat – using synthetic GCaMP6f kernel instead.")
    t_k           = np.arange(kernel_win_frames) / framerate
    tau_rise, tau_decay = 0.15, 0.6
    kernel_raw    = (1 - np.exp(-t_k / tau_rise)) * np.exp(-t_k / tau_decay)
    calcium_kernel = kernel_raw / kernel_raw.sum()

# Kernel time axis
t_kernel = np.arange(len(calcium_kernel)) / framerate

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t_kernel, calcium_kernel, color='crimson', lw=2)
ax.set_xlabel('Time after event (s)')
ax.set_ylabel('Kernel weight (a.u.)')
ax.set_title('Estimated GCaMP calcium kernel (reward-triggered average)')
ax.axvline(0, color='k', lw=0.8, ls='--')
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════════
# 2.2  Raised cosine spatial basis functions
# ════════════════════════════════════════════════════════════════════════
N_SPATIAL_BASIS = 10          # number of basis functions
pos_min, pos_max = 0.0, TRACK_LENGTH_CM

# Centre and width of each cosine bump
basis_centres = np.linspace(pos_min, pos_max, N_SPATIAL_BASIS)
basis_width   = 2.0 * (basis_centres[1] - basis_centres[0])  # full-width, 2× spacing

def raised_cosine_basis(position, centres, width):
    """Evaluate N raised cosine basis functions at each frame.
    Returns array (n_frames, N_basis).
    """
    n_frames = len(position)
    n_basis  = len(centres)
    B        = np.zeros((n_frames, n_basis))
    for j, c in enumerate(centres):
        diff   = position - c
        in_win = np.abs(diff) < width / 2
        B[in_win, j] = 0.5 * (1 + np.cos(2 * np.pi * diff[in_win] / width))
    return B

X_spatial = raised_cosine_basis(location_cm, basis_centres, basis_width)
print(f"Spatial basis: {X_spatial.shape}  (n_frames × {N_SPATIAL_BASIS} basis fns)")

# ════════════════════════════════════════════════════════════════════════
# 2.3  Reward-proximity regressors
# ════════════════════════════════════════════════════════════════════════
# Distance from reward zone (REWARD_ZONE_CM), restricted to [0, TRACK_LENGTH_CM]
dist_to_reward = np.abs(location_cm - REWARD_ZONE_CM)          # raw distance

# Normalise to [0, 1] so linear and quadratic terms are comparable
dist_norm  = dist_to_reward / TRACK_LENGTH_CM

X_reward = np.column_stack([
    dist_norm,           # linear distance (large = far from reward)
    dist_norm ** 2,      # quadratic distance
])
print(f"Reward-proximity regressors: {X_reward.shape}")

# ════════════════════════════════════════════════════════════════════════
# 2.4  Landmark onset regressors (convolved with calcium kernel)
# ════════════════════════════════════════════════════════════════════════
# Detect each frame where the animal first crosses each landmark position.
# Within each lap, we detect a single crossing per landmark.

def detect_landmark_crossings(location_cm, lap_starts, lap_ends, landmark_cm,
                               crossing_tolerance_cm=5.0):
    """Return frame indices where the animal crosses `landmark_cm` (one per lap)."""
    crossing_frames = []
    for ls, le in zip(lap_starts, lap_ends):
        lap_loc = location_cm[ls:le]
        # Find first frame where position is within tolerance of landmark
        near   = np.where(np.abs(lap_loc - landmark_cm) <= crossing_tolerance_cm)[0]
        if len(near) > 0:
            crossing_frames.append(ls + near[0])
    return np.array(crossing_frames, dtype=int)

X_landmark_raw = np.zeros((n_frames, len(LANDMARK_POSITIONS)))
landmark_crossing_frames = []

for j, lm_cm in enumerate(LANDMARK_POSITIONS):
    crossings = detect_landmark_crossings(
        location_cm, lap_starts, lap_ends, lm_cm, crossing_tolerance_cm=4.0
    )
    landmark_crossing_frames.append(crossings)
    X_landmark_raw[crossings, j] = 1.0   # impulse at crossing frame

print(f"Landmark crossings detected: "
      + ", ".join([f"LM{i+1}@{p}cm={len(c)}" 
                   for i, (p, c) in enumerate(
                       zip(LANDMARK_POSITIONS, landmark_crossing_frames))]))

# Convolve each impulse train with the calcium kernel
X_landmark = np.zeros_like(X_landmark_raw)
for j in range(len(LANDMARK_POSITIONS)):
    X_landmark[:, j] = convolve(X_landmark_raw[:, j],
                                 calcium_kernel,
                                 mode='full')[:n_frames]

print(f"Landmark regressors (convolved): {X_landmark.shape}")

# ════════════════════════════════════════════════════════════════════════
# 2.5  Nuisance regressors
# ════════════════════════════════════════════════════════════════════════

def z_score(x):
    return (x - x.mean()) / (x.std() + 1e-8)

# Running speed (z-scored)
speed_z = z_score(speed)

# Impulse at trial onset (lap start) and trial offset (lap end)
trial_onset_impulse  = np.zeros(n_frames)
trial_offset_impulse = np.zeros(n_frames)
reward_impulse       = np.zeros(n_frames)

trial_onset_impulse[lap_starts]   = 1.0
trial_offset_impulse[lap_ends - 1] = 1.0

# Reward impulse (convolve with calcium kernel)
reward_impulse_raw = np.zeros(n_frames)
valid_reward = reward_frames[reward_frames < n_frames]
reward_impulse_raw[valid_reward] = 1.0
reward_impulse = convolve(reward_impulse_raw, calcium_kernel, mode='full')[:n_frames]

X_nuisance = np.column_stack([
    speed_z,
    trial_onset_impulse,
    trial_offset_impulse,
    reward_impulse,
])

# ════════════════════════════════════════════════════════════════════════
# 2.6  Assemble full design matrix
# ════════════════════════════════════════════════════════════════════════
# Add a constant (intercept) column
X_intercept = np.ones((n_frames, 1))

X = np.column_stack([X_intercept, X_spatial, X_reward, X_landmark, X_nuisance])

# Record which column indices belong to each group
n_intercept = 1
n_spatial   = N_SPATIAL_BASIS
n_reward    = 2
n_landmark  = len(LANDMARK_POSITIONS)
n_nuisance  = 4

col_start = {}
col_start['intercept'] = 0
col_start['spatial']   = n_intercept
col_start['reward']    = col_start['spatial']   + n_spatial
col_start['landmark']  = col_start['reward']    + n_reward
col_start['nuisance']  = col_start['landmark']  + n_landmark

regressor_groups = {
    'spatial' : list(range(col_start['spatial'],  col_start['spatial']  + n_spatial)),
    'reward'  : list(range(col_start['reward'],   col_start['reward']   + n_reward)),
    'landmark': list(range(col_start['landmark'], col_start['landmark'] + n_landmark)),
    'nuisance': list(range(col_start['nuisance'], col_start['nuisance'] + n_nuisance)),
}
regressor_names = (
    ['intercept']
    + [f'spatial_{i}' for i in range(n_spatial)]
    + ['reward_lin', 'reward_quad']
    + [f'LM{i+1}_{p}cm' for i, p in enumerate(LANDMARK_POSITIONS)]
    + ['speed_z', 'trial_onset', 'trial_offset', 'reward_impulse']
)

print(f"\nDesign matrix X: {X.shape}  (n_frames × n_regressors)")
print(f"  Intercept  cols: {col_start['intercept']}")
print(f"  Spatial    cols: {regressor_groups['spatial'][0]}–{regressor_groups['spatial'][-1]}")
print(f"  Reward     cols: {regressor_groups['reward'][0]}–{regressor_groups['reward'][-1]}")
print(f"  Landmark   cols: {regressor_groups['landmark'][0]}–{regressor_groups['landmark'][-1]}")
print(f"  Nuisance   cols: {regressor_groups['nuisance'][0]}–{regressor_groups['nuisance'][-1]}")

# Quick sanity: check for NaN/Inf
assert not np.any(np.isnan(X)), "NaN in design matrix!"
assert not np.any(np.isinf(X)), "Inf in design matrix!"
print("\n✓ Design matrix has no NaN or Inf.")


---
## CHUNK 3 — Ridge regression with cross-validation

For each cell we fit four models and record R² on held-out data:

| Model | Regressors included |
|---|---|
| Full | intercept + spatial + reward + landmark + nuisance |
| Reduced-spatial | full minus spatial |
| Reduced-reward | full minus reward |
| Reduced-landmark | full minus landmark |

Ridge parameter λ is selected per cell via nested 5-fold cross-validation
(`RidgeCV`) on the **full model** and reused for reduced models.

**Why ridge?** Raised cosine basis functions are correlated (overlapping support)
and the landmark/reward regressors are also collinear with the spatial basis.
Ridge regularisation handles this gracefully while remaining analytically
tractable.

**Why deconvolved spikes as target?** Deconvolved spikes remove the slow
GCaMP decay, making the GLM more temporally precise.  We already modelled the
GCaMP kernel explicitly in the regressors (Chunk 2), so this is consistent.


In [ ]:
from sklearn.linear_model  import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics         import r2_score

# ─── Settings ────────────────────────────────────────────────────────────────
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
N_CV_FOLDS   = 5

# ─── Helper: fit one model and return cross-validated R² ─────────────────────
def fit_ridge_cv(X_model, y, alphas, cv):
    """Fit RidgeCV and return cross-validated R² (mean over folds).

    We fit the ridge on training data and evaluate R² on held-out data to
    get an unbiased estimate of explained variance.  RidgeCV internally
    selects the best alpha using leave-one-out on the training fold.
    """
    r2_folds    = []
    best_alphas = []

    for train_idx, test_idx in cv.split(X_model):
        X_train, X_test = X_model[train_idx], X_model[test_idx]
        y_train, y_test = y[train_idx],       y[test_idx]

        model = RidgeCV(alphas=alphas, fit_intercept=False)
        model.fit(X_train, y_train)

        y_pred   = model.predict(X_test)
        r2       = r2_score(y_test, y_pred)
        r2_folds.append(r2)
        best_alphas.append(model.alpha_)

    return np.mean(r2_folds), np.median(best_alphas)


# ─── Column masks for reduced models ─────────────────────────────────────────
all_cols  = list(range(X.shape[1]))
reduced_col_masks = {
    'full'    : all_cols,
    'no_spatial'  : [c for c in all_cols if c not in regressor_groups['spatial']],
    'no_reward'   : [c for c in all_cols if c not in regressor_groups['reward']],
    'no_landmark' : [c for c in all_cols if c not in regressor_groups['landmark']],
}

# ─── Main loop ───────────────────────────────────────────────────────────────
cv = KFold(n_splits=N_CV_FOLDS, shuffle=False)

r2_full     = np.zeros(n_cells)
r2_no_sp    = np.zeros(n_cells)   # full without spatial
r2_no_rw    = np.zeros(n_cells)   # full without reward
r2_no_lm    = np.zeros(n_cells)   # full without landmark
best_alpha  = np.zeros(n_cells)

print(f"Fitting GLM for {n_cells} cells × 4 models × {N_CV_FOLDS} folds …")

for cell_idx in range(n_cells):
    y = spikes[cell_idx, :]  # (n_frames,)

    # Skip silent cells
    if y.max() == 0 or y.std() == 0:
        continue

    r2_full[cell_idx],  alpha = fit_ridge_cv(X[:, reduced_col_masks['full']],    y, RIDGE_ALPHAS, cv)
    r2_no_sp[cell_idx], _     = fit_ridge_cv(X[:, reduced_col_masks['no_spatial']],  y, [alpha], cv)
    r2_no_rw[cell_idx], _     = fit_ridge_cv(X[:, reduced_col_masks['no_reward']],   y, [alpha], cv)
    r2_no_lm[cell_idx], _     = fit_ridge_cv(X[:, reduced_col_masks['no_landmark']], y, [alpha], cv)
    best_alpha[cell_idx]      = alpha

    if (cell_idx + 1) % 50 == 0 or cell_idx == 0:
        print(f"  cell {cell_idx+1:4d}/{n_cells} – "
              f"R²_full={r2_full[cell_idx]:.3f}, alpha={alpha:.3f}")

print("\nDone.")
print(f"Median R²_full (all cells): {np.median(r2_full):.3f}")
print(f"Cells with R²_full > 0.05  : {(r2_full > 0.05).sum()} / {n_cells}")


---
## CHUNK 4 — Partial R² and cell classification

**Partial R²** for each regressor group is the drop in R² when that group
is removed from the full model:

$$\text{partial-}R^2_{\text{group}} = R^2_{\text{full}} - R^2_{\text{full} \setminus \text{group}}$$

A positive value means the group adds unique explanatory power *above and
beyond* all other regressors (including nuisance variables).

**Classification rules**
- A cell is considered *responsive* if `R²_full > R2_FULL_THRESHOLD` (0.05)
- Among responsive cells, the group with the **highest partial R²** (if it
  exceeds `PARTIAL_R2_THRESHOLD`, default 0.02) determines the label:

| Label | Condition |
|---|---|
| `spatial` | spatial partial R² is highest & > threshold |
| `reward-proximity` | reward partial R² is highest & > threshold |
| `landmark` | landmark partial R² is highest & > threshold |
| `mixed` | ≥ 2 groups exceed threshold |
| `unclassified` | R²_full < threshold OR all partial R² < threshold |

**Why 0.02?**  Based on simulations with Poisson-like noise at typical GCaMP
signal levels, a partial R² of 0.02 corresponds roughly to a 5% false-positive
rate.  Adjust based on your data noise level.


In [ ]:
# ─── Thresholds ──────────────────────────────────────────────────────────────
R2_FULL_THRESHOLD    = 0.05   # cell must have some overall predictability
PARTIAL_R2_THRESHOLD = 0.02   # minimum unique contribution to count

# ─── Compute partial R² per group ────────────────────────────────────────────
partial_r2 = {
    'spatial' : np.clip(r2_full - r2_no_sp, 0, None),
    'reward'  : np.clip(r2_full - r2_no_rw, 0, None),
    'landmark': np.clip(r2_full - r2_no_lm, 0, None),
}
# Stack for convenience
partial_r2_mat = np.column_stack([
    partial_r2['spatial'],
    partial_r2['reward'],
    partial_r2['landmark'],
])  # (n_cells, 3)

GROUP_NAMES  = ['spatial', 'reward', 'landmark']
LABEL_NAMES  = ['spatial', 'reward-proximity', 'landmark', 'mixed', 'unclassified']
LABEL_COLORS = {
    'spatial'          : '#2196F3',  # blue
    'reward-proximity' : '#F44336',  # red
    'landmark'         : '#4CAF50',  # green
    'mixed'            : '#FF9800',  # orange
    'unclassified'     : '#9E9E9E',  # grey
}

# ─── Classify each cell ───────────────────────────────────────────────────────
cell_labels = np.array(['unclassified'] * n_cells, dtype=object)

for i in range(n_cells):
    # Must meet overall R² threshold
    if r2_full[i] < R2_FULL_THRESHOLD:
        continue  # stays unclassified

    # Count how many groups exceed partial R² threshold
    groups_above = partial_r2_mat[i] > PARTIAL_R2_THRESHOLD

    if groups_above.sum() == 0:
        continue  # stays unclassified
    elif groups_above.sum() >= 2:
        cell_labels[i] = 'mixed'
    else:
        # Single dominant group
        best_group_idx = np.argmax(partial_r2_mat[i])
        label_map      = {0: 'spatial', 1: 'reward-proximity', 2: 'landmark'}
        cell_labels[i] = label_map[best_group_idx]

# ─── Summary ─────────────────────────────────────────────────────────────────
print("Cell classification summary:")
for lab in LABEL_NAMES:
    n = (cell_labels == lab).sum()
    print(f"  {lab:20s}: {n:4d}  ({100*n/n_cells:.1f}%)")

print(f"\nPartial R² statistics (responsive cells, R²_full > {R2_FULL_THRESHOLD}):")
resp = r2_full > R2_FULL_THRESHOLD
for grp in GROUP_NAMES:
    pr = partial_r2[grp][resp]
    print(f"  {grp:10s}: mean={pr.mean():.3f}, median={np.median(pr):.3f}, "
          f"max={pr.max():.3f}")

# ─── Partial R² scatter: reward-proximity vs spatial ────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
for lab in LABEL_NAMES:
    mask = cell_labels == lab
    ax.scatter(
        partial_r2['spatial'][mask],
        partial_r2['reward'][mask],
        s=15, alpha=0.6,
        color=LABEL_COLORS[lab],
        label=f"{lab} (n={mask.sum()})",
        zorder=3,
    )
ax.axhline(PARTIAL_R2_THRESHOLD, color='k', lw=0.8, ls='--', alpha=0.5)
ax.axvline(PARTIAL_R2_THRESHOLD, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Partial R² (spatial)')
ax.set_ylabel('Partial R² (reward-proximity)')
ax.set_title('Spatial vs reward-proximity: partial R²')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()


---
## CHUNK 5 — Layer-specific summary statistics and plots

Four figures:
1. Proportion of each cell type per cortical layer (stacked bar)
2. Partial R² scatter (reward-proximity vs spatial), coloured by layer
3. Developmental trajectory: cell type proportions per layer across training days 1–7
4. Example tuning curves: top spatial cells vs top reward-proximity cells per layer

Layer assignment replicates the logic in `helper/SpatialModulationIndexLayerSpecific.py`:
peak cell density along the y-axis = L4 centre; ±70 µm = L4 boundaries;
L5 extends 150 µm below L4; L2/3 is above L4; L6 is below L5.


In [ ]:
import seaborn as sns
from scipy.ndimage import gaussian_filter1d

# ─────────────────────────────────────────────────────────────────────────────
# 5.0  Layer assignment
# ─────────────────────────────────────────────────────────────────────────────
UM_PER_PIXEL  = 0.947408849697405  # at 1.15× objective
L4_HALF_UM    = 70.0               # L4 = peak ± 70 µm
L5_OFFSET_UM  = 150.0              # L5 lower edge = L4 lower + 150 µm

L4_HALF_PIX   = L4_HALF_UM  / UM_PER_PIXEL
L5_OFFSET_PIX = L5_OFFSET_UM / UM_PER_PIXEL

y_coords = med_coords[:, 0]   # y pixel coordinate per cell

# Detect L4 from peak of cell density histogram
hist, bin_edges = np.histogram(y_coords, bins=50)
hist_smooth     = gaussian_filter1d(hist.astype(float), sigma=1)
l4_center_pix   = (bin_edges[np.argmax(hist_smooth)] + bin_edges[np.argmax(hist_smooth) + 1]) / 2

l4_upper  = l4_center_pix - L4_HALF_PIX
l4_lower  = l4_center_pix + L4_HALF_PIX
l5_upper  = l4_lower
l5_lower  = l4_lower + L5_OFFSET_PIX

LAYER_BOUNDS = {
    'L2/3': (0,         l4_upper),
    'L4'  : (l4_upper,  l4_lower),
    'L5'  : (l5_upper,  l5_lower),
    'L6'  : (l5_lower,  np.inf),
}
LAYER_COLORS = {'L2/3': '#E91E63', 'L4': '#9C27B0', 'L5': '#2196F3', 'L6': '#009688'}
LAYER_ORDER  = ['L2/3', 'L4', 'L5', 'L6']

layer_masks = {}
for layer, (lo, hi) in LAYER_BOUNDS.items():
    layer_masks[layer] = (y_coords >= lo) & (y_coords < hi)

print("Cells per layer:")
for layer in LAYER_ORDER:
    print(f"  {layer}: {layer_masks[layer].sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# 5.1  Figure 1: Stacked bar — cell type proportions per layer
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
x_pos = np.arange(len(LAYER_ORDER))
bottoms = np.zeros(len(LAYER_ORDER))

for lab in LABEL_NAMES:
    props = []
    for layer in LAYER_ORDER:
        mask = layer_masks[layer]
        n_layer = mask.sum()
        n_type  = (cell_labels[mask] == lab).sum()
        props.append(n_type / n_layer if n_layer > 0 else 0)
    ax.bar(x_pos, props, bottom=bottoms,
           color=LABEL_COLORS[lab], label=lab, width=0.6)
    bottoms += np.array(props)

ax.set_xticks(x_pos)
ax.set_xticklabels(LAYER_ORDER)
ax.set_ylabel('Proportion of cells')
ax.set_ylim(0, 1)
ax.set_title('Cell type proportions by layer')
ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.3, 1))
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 5.2  Figure 2: Partial R² scatter coloured by layer
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
for layer in LAYER_ORDER:
    mask = layer_masks[layer]
    ax.scatter(
        partial_r2['spatial'][mask],
        partial_r2['reward'][mask],
        s=12, alpha=0.5,
        color=LAYER_COLORS[layer],
        label=f"{layer} (n={mask.sum()})",
    )
ax.axhline(PARTIAL_R2_THRESHOLD, color='k', lw=0.8, ls='--', alpha=0.5)
ax.axvline(PARTIAL_R2_THRESHOLD, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Partial R² (spatial)')
ax.set_ylabel('Partial R² (reward-proximity)')
ax.set_title('Spatial vs reward-proximity by layer')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 5.3  Figure 3: Developmental trajectory across days 1–7
# ─────────────────────────────────────────────────────────────────────────────
# ⚠ REQUIRES: cell_labels_by_day, layer_masks_by_day computed from each day's HDF5.
# The block below shows the template; fill in PREPROC_H5_BY_DAY with your paths.

PREPROC_H5_BY_DAY = {
    # day: path_to_preproc_h5
    # 1: r"D:\...\251009_JSY052_preproc.h5",
    # 2: r"D:\...\251010_JSY052_preproc.h5",
    # ... fill in for days 1–7
}

if len(PREPROC_H5_BY_DAY) > 0:
    # Re-run Chunks 1–4 for each day and collect cell_labels + layer_masks
    # This block is a sketch; adapt as needed.
    from types import SimpleNamespace

    day_results = {}  # day → SimpleNamespace with .cell_labels, .layer_masks
    # <run GLM pipeline per day here>

    # Plotting
    days = sorted(PREPROC_H5_BY_DAY.keys())
    fig, axes = plt.subplots(1, len(LAYER_ORDER), figsize=(14, 4), sharey=True)

    for ax, layer in zip(axes, LAYER_ORDER):
        for lab in ['spatial', 'reward-proximity', 'landmark', 'mixed']:
            props = []
            for d in days:
                res  = day_results[d]
                mask = res.layer_masks[layer]
                n    = mask.sum()
                props.append((res.cell_labels[mask] == lab).sum() / n if n > 0 else 0)
            ax.plot(days, props, marker='o', color=LABEL_COLORS[lab], label=lab)
        ax.set_title(layer)
        ax.set_xlabel('Training day')
        ax.set_xticks(days)
        if ax == axes[0]:
            ax.set_ylabel('Proportion')
            ax.legend(fontsize=7)
    fig.suptitle('Developmental trajectory of cell type proportions', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("[Figure 3 skipped] Fill in PREPROC_H5_BY_DAY to enable developmental trajectory.")

# ─────────────────────────────────────────────────────────────────────────────
# 5.4  Figure 4: Example tuning curves — top spatial vs top reward-proximity
# ─────────────────────────────────────────────────────────────────────────────
N_EXAMPLES = 3  # examples per cell type per layer

def compute_tuning_curve(spikes_cell, location_cm, bin_centers, n_laps):
    """Mean activity per spatial bin (trial-averaged)."""
    n_bins   = len(bin_centers)
    bin_size = bin_centers[1] - bin_centers[0]
    tc = np.zeros(n_bins)
    for b, pos in enumerate(bin_centers):
        in_bin = np.abs(location_cm - pos) < bin_size / 2
        if in_bin.sum() > 0:
            tc[b] = spikes_cell[in_bin].mean()
    # Normalise to [0, 1]
    if tc.max() > 0:
        tc = tc / tc.max()
    return tc

fig, axes = plt.subplots(
    len(LAYER_ORDER), 2 * N_EXAMPLES,
    figsize=(4 * N_EXAMPLES, 3 * len(LAYER_ORDER)),
    sharey=True
)

for row, layer in enumerate(LAYER_ORDER):
    mask = layer_masks[layer]

    # Top spatial cells in this layer
    sp_scores = np.where(mask, partial_r2['spatial'], -np.inf)
    top_spatial = np.argsort(sp_scores)[::-1][:N_EXAMPLES]

    # Top reward-proximity cells in this layer
    rw_scores = np.where(mask, partial_r2['reward'], -np.inf)
    top_reward = np.argsort(rw_scores)[::-1][:N_EXAMPLES]

    for col_i, cell_idx in enumerate(top_spatial):
        ax = axes[row, col_i]
        tc = compute_tuning_curve(spikes[cell_idx], location_cm, bin_centers, n_laps)
        ax.plot(bin_centers, tc, color='#2196F3', lw=1.5)
        for lm in LANDMARK_POSITIONS:
            ax.axvline(lm, color='gray', lw=0.6, ls='--', alpha=0.5)
        ax.axvline(REWARD_ZONE_CM, color='red', lw=0.8, ls='-', alpha=0.5)
        ax.set_title(f"{layer} Spatial\ncell {cell_idx} R²={partial_r2['spatial'][cell_idx]:.2f}",
                     fontsize=7)
        if col_i == 0:
            ax.set_ylabel('Norm. activity')

    for col_i, cell_idx in enumerate(top_reward):
        ax = axes[row, N_EXAMPLES + col_i]
        tc = compute_tuning_curve(spikes[cell_idx], location_cm, bin_centers, n_laps)
        ax.plot(bin_centers, tc, color='#F44336', lw=1.5)
        for lm in LANDMARK_POSITIONS:
            ax.axvline(lm, color='gray', lw=0.6, ls='--', alpha=0.5)
        ax.axvline(REWARD_ZONE_CM, color='red', lw=0.8, ls='-', alpha=0.5)
        ax.set_title(f"{layer} Reward-prox\ncell {cell_idx} R²={partial_r2['reward'][cell_idx]:.2f}",
                     fontsize=7)

for ax in axes[-1, :]:
    ax.set_xlabel('Position (cm)')

fig.suptitle(
    'Example tuning curves: top spatial (blue) vs reward-proximity (red) cells',
    fontsize=10
)
plt.tight_layout()
plt.show()
